# Training ResNet-18 Model

The goal of this notebook is to allow for a user to be able to train a ResNet-18 Model. 
The data collection notebook and the data augmentation notebook should have been done before completing this notebook. 

We will want a ResNet-18 Model because it is extremely light weight and can easily run on the Jetson Orin Nano to be able to have other functionalities on the device. 

The device I will use to train and do the work on first pass is a Macbook M3 Pro with 18 gb RAM to just get it to work first and fast iteration 

#### Important Imports

In [1]:
# --- One-cell, notebook-friendly ResNet18 trainer (fixed AMP + scheduler + multiprocessing) ---

import os, sys, time, math, random, json
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, Subset, random_split

import torchvision
from torchvision import transforms, datasets

# -----------------------------
# Config (edit here in-notebook)
# -----------------------------
CFG = dict(
    data_dir="LABEL_AND_ALL_DATA/",     # Path with either train/val subdirs, or a single folder to be split
    outdir="outputs",
    batch_size=64,
    epochs=15,
    lr=3e-4,
    weight_decay=1e-4,
    train_ratio=0.8,               # used only if we need to split a single folder
    img_size=400,
    num_workers=2,                 # will be forced to 0 inside notebooks to avoid pickling issues
    seed=42,
    pretrained=True,               # use ImageNet weights
    label_smoothing=0.0
)

# -----------------------------
# Repro & device
# -----------------------------
def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # torch.use_deterministic_algorithms(True)  # enable if you need strict determinism (may slow things)

set_seed(CFG["seed"])

device = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)

use_autocast = device.type in ("cuda", "mps")
scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None
if device.type == "mps":
    torch.set_float32_matmul_precision("medium")

print(f"Using device: {device}")

# -----------------------------
# Data: transforms & datasets
# -----------------------------
IMG_SIZE = CFG["img_size"]
train_tfms = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.15)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])
val_tfms = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.15)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

data_dir = Path(CFG["data_dir"])
if not data_dir.exists():
    raise FileNotFoundError(f"Data directory not found: {data_dir.resolve()}")

train_dir = data_dir / "train"
val_dir = data_dir / "val"

if train_dir.exists() and val_dir.exists():
    train_ds = datasets.ImageFolder(train_dir, transform=train_tfms)
    val_ds   = datasets.ImageFolder(val_dir,   transform=val_tfms)
else:
    # Single folder: split into train/val using ImageFolder once
    full_ds = datasets.ImageFolder(data_dir, transform=train_tfms)
    n_total = len(full_ds)
    n_train = int(n_total * CFG["train_ratio"])
    n_val = n_total - n_train
    set_seed(CFG["seed"])
    train_subset, val_subset = random_split(full_ds, [n_train, n_val])
    # Ensure val uses val_tfms (wrap via Subset with a view that swaps transform at dataset level)
    # Simpler: create two datasets pointing to same folder, but index via Subset indices
    full_train = datasets.ImageFolder(data_dir, transform=train_tfms)
    full_val   = datasets.ImageFolder(data_dir, transform=val_tfms)
    train_ds = Subset(full_train, train_subset.indices)
    val_ds   = Subset(full_val,   val_subset.indices)

num_classes = len(train_ds.dataset.classes if isinstance(train_ds, Subset) else train_ds.classes)
class_names = (train_ds.dataset.classes if isinstance(train_ds, Subset) else train_ds.classes)
print(f"Classes ({num_classes}): {class_names}")

# Notebook-safe workers: avoid pickling custom locals
in_notebook = "ipykernel" in sys.modules
num_workers = 0 if in_notebook else CFG["num_workers"]

pin_memory = (device.type == "cuda")
train_loader = DataLoader(
    train_ds, batch_size=CFG["batch_size"], shuffle=True,
    num_workers=num_workers, persistent_workers=False, pin_memory=pin_memory
)
val_loader = DataLoader(
    val_ds, batch_size=CFG["batch_size"], shuffle=False,
    num_workers=num_workers, persistent_workers=False, pin_memory=pin_memory
)

# -----------------------------
# Model
# -----------------------------
if CFG["pretrained"]:
    try:
        weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
    except AttributeError:
        # Fallback for older/newer torchvision enums
        weights = "IMAGENET1K_V1"
    model = torchvision.models.resnet18(weights=weights)
else:
    model = torchvision.models.resnet18(weights=None)

# Replace classifier
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(in_features, num_classes)
)
model.to(device)

# -----------------------------
# Optimizer, Scheduler, Loss
# -----------------------------
optimizer = optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["epochs"])
criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"]).to(device)

# -----------------------------
# Train / Eval loops
# -----------------------------
def train_one_epoch(model, loader, optimizer, device, scaler):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, targets in loader:
        images = images.to(device, non_blocking=False)
        targets = targets.to(device, non_blocking=False)

        optimizer.zero_grad(set_to_none=True)

        if use_autocast:
            with torch.amp.autocast(device_type=device.type, dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, targets)
        else:
            outputs = model(images)
            loss = criterion(outputs, targets)

        if scaler is not None:           # CUDA path
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:                             # CPU/MPS path
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    return running_loss / max(1, total), correct / max(1, total)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, targets in loader:
        images = images.to(device, non_blocking=False)
        targets = targets.to(device, non_blocking=False)

        if use_autocast:
            with torch.amp.autocast(device_type=device.type, dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, targets)
        else:
            outputs = model(images)
            loss = criterion(outputs, targets)

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    return running_loss / max(1, total), correct / max(1, total)

# -----------------------------
# Train
# -----------------------------
os.makedirs(CFG["outdir"], exist_ok=True)
best_val_acc = -1.0
best_path = str(Path(CFG["outdir"]) / "resnet18_best.pth")

start_time = time.time()
for epoch in range(1, CFG["epochs"] + 1):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, device, scaler)
    val_loss, val_acc = evaluate(model, val_loader, device)
    scheduler.step()

    dt = time.time() - t0
    print(f"[{epoch:03d}/{CFG['epochs']}] "
          f"train_loss={train_loss:.4f} acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} acc={val_acc:.4f} | {dt:.1f}s")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "epoch": epoch,
            "val_acc": best_val_acc,
            "class_names": class_names,
            "cfg": CFG
        }, best_path)

total_time = time.time() - start_time
print(f"Done. Best val acc: {best_val_acc:.4f}. Saved: {best_path}. Total time: {total_time/60:.1f} min")


/Users/erickortega/venv3.12/lib/python3.12/site-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/Context.cpp:85.)
  _C._set_float32_matmul_precision(precision)


Using device: mps
Classes (8): ['Bad Hand View', 'Nothing', 'five ', 'four', 'one', 'three', 'two ', 'zero']
[001/15] train_loss=0.6640 acc=0.7816 | val_loss=0.3786 acc=0.8687 | 138.3s
[002/15] train_loss=0.2351 acc=0.9280 | val_loss=0.2765 acc=0.9125 | 137.2s
[003/15] train_loss=0.1566 acc=0.9508 | val_loss=0.2670 acc=0.9242 | 137.0s
[004/15] train_loss=0.0953 acc=0.9704 | val_loss=0.2219 acc=0.9379 | 139.8s
[005/15] train_loss=0.0763 acc=0.9757 | val_loss=0.1751 acc=0.9438 | 143.5s
[006/15] train_loss=0.0382 acc=0.9892 | val_loss=0.1825 acc=0.9445 | 138.0s
[007/15] train_loss=0.0248 acc=0.9925 | val_loss=0.1976 acc=0.9432 | 137.6s
[008/15] train_loss=0.0196 acc=0.9954 | val_loss=0.2023 acc=0.9458 | 136.7s
[009/15] train_loss=0.0121 acc=0.9982 | val_loss=0.1682 acc=0.9562 | 137.1s
[010/15] train_loss=0.0052 acc=0.9995 | val_loss=0.1577 acc=0.9608 | 136.8s
[011/15] train_loss=0.0038 acc=0.9997 | val_loss=0.1586 acc=0.9621 | 136.7s
[012/15] train_loss=0.0025 acc=0.9998 | val_loss=0.1555

In [5]:
# --- TorchScript export (robust to fc / fc.0 key layout) ---

import torch, torchvision
from pathlib import Path

# Paths derived from your CFG (matches your first snippet)
RUN_DIR = Path(CFG["outdir"]) / f"{CFG['img_size']}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
best_path   = RUN_DIR / "resnet18_best.pth"
labels_path = RUN_DIR / "labels.txt"
ts_path     = RUN_DIR / "resnet18_scripted.pt"

# Load the best checkpoint on CPU
state = torch.load(best_path, map_location="cpu")

# Class names and count
class_names = state.get("class_names", None)
if class_names is None:
    raise KeyError("Checkpoint missing 'class_names'. Re-train or add it before saving.")
num_classes = len(class_names)

# Rebuild base model on CPU (no pretrained weights needed for export)
model_cpu = torchvision.models.resnet18(weights=None)
in_features = model_cpu.fc.in_features

# Detect how the head was defined during training, and rebuild to match
state_dict = state["model_state"]
has_seq_head = any(k.startswith("fc.0.") for k in state_dict.keys())

if has_seq_head:
    # Training used: model.fc = nn.Sequential(nn.Linear(...))
    model_cpu.fc = torch.nn.Sequential(torch.nn.Linear(in_features, num_classes))
else:
    # Training used: model.fc = nn.Linear(...)
    model_cpu.fc = torch.nn.Linear(in_features, num_classes)

# Try strict load; if that fails due to minor naming diffs, fall back to a safe remap
try:
    model_cpu.load_state_dict(state_dict, strict=True)
except RuntimeError as e:
    # Handle the common mismatch: convert fc.0.* <-> fc.*
    remapped = {}
    for k, v in state_dict.items():
        if k.startswith("fc.0."):
            remapped[k.replace("fc.0.", "fc.")] = v
        elif k.startswith("fc.") and "weight" in k or "bias" in k:
            # If the model has Sequential head, map fc.* -> fc.0.*
            if isinstance(model_cpu.fc, torch.nn.Sequential):
                remapped[k.replace("fc.", "fc.0.")] = v
            else:
                remapped[k] = v
        else:
            remapped[k] = v
    model_cpu.load_state_dict(remapped, strict=False)  # allow unrelated buffers if any

# Eval on CPU and export TorchScript
model_cpu.eval().cpu()

# (Optional) Small dummy run to ensure graph is traceable with your head shape
with torch.inference_mode():
    _ = model_cpu(torch.randn(1, 3, CFG["img_size"], CFG["img_size"]))

# Prefer scripting (works with simple control flow); for pure feed-forward this is fine
scripted = torch.jit.script(model_cpu)
scripted.save(str(ts_path))

# Ensure labels.txt sits next to the exported model
with open(labels_path, "w", encoding="utf-8") as f:
    for name in class_names:
        f.write(str(name).strip() + "\n")

print(f"TorchScript saved to: {ts_path}")
print(f"Labels written to:   {labels_path}")


TorchScript saved to: outputs/400/resnet18_scripted.pt
Labels written to:   outputs/400/labels.txt
